In [172]:

import pickle
import numpy as np
from pathlib import Path
from tqdm import tqdm
from vebir.pca import loo_pca, malinowski_ind
from vebir.ebs import EBS
import pandas as pd 
import matplotlib.pyplot as plt

In [173]:
def compute_cors(compound,params,ref_dict,corrections_dict,wn):
    corrections = corrections_dict[compound][params] 
    ref_profile = ref_dict[compound][:,1]
    ref_wn = ref_dict[compound][:,0]
    interpolated_corrections = [np.interp(ref_wn, wn, c) for c in corrections]
    correlations = [np.corrcoef(ref_profile, c)[0, 1] for c in interpolated_corrections]
    return np.array(correlations) 

In [174]:
REPO_ROOT = Path.cwd().resolve().parent
CORRECTIONS_DIR = REPO_ROOT / "data" / "corrections" / "teflon"
GT_DIR = REPO_ROOT / "data" / "spectrabase" / "teflon"
LAB_DIR = REPO_ROOT / "data" / "raw" / "teflon" / "laboratory_samples"
PREPROC_DIR = REPO_ROOT / "data" / "preprocessed" / "teflon"

with open(PREPROC_DIR / "blanks_dict.pkl", "rb") as f:
    blanks_dict = pickle.load(f)

with open(GT_DIR / "ref_dict.pkl", 'rb') as file:
        ref_dict = pickle.load(file) 

Z = np.array(blanks_dict["2011"])
    
wn = np.sort(pd.read_csv(LAB_DIR / "zerofilling.txt", header=None).iloc[:, 0])
compounds = ['12-Tricosanone', 'Ammonium sulfate', 'Malonic Acid', 'Suberic Acid', 'D-Glucose', 'fructose', 'levoglucosan']

In [234]:
chat_ind = malinowski_ind(Z)[0]
chat_cv = loo_pca(Z)[1]
print(f"IND: {chat_ind}, LOO-OSE: {chat_cv}")


IND: 53, LOO-OSE: 47


In [226]:
df = {}

In [260]:
with open(CORRECTIONS_DIR / "ebs_als_dict_W.pkl", "rb") as f:
    corrections_dict = pickle.load(f)

In [275]:
def ebs_metrics(corrections_dict, ref_dict, wn, chat, compounds, parameterization = "V", c_estimation="IND"):
    df = {}
    
    params = f"ncomp,tau = {chat},{0.10}"
    means = []
    se = []
    all_correlations = []

    for compound in compounds:
        correlations = compute_cors(compound,params,ref_dict,corrections_dict,wn)
        all_correlations.append(correlations)
        means.append(100 * np.mean(correlations))
        se.append(100 * np.std(correlations) / np.sqrt(len(correlations)))
    all_correlations = np.concatenate(all_correlations)
    means.append(100 * np.mean(all_correlations))
    se.append(100 * np.std(all_correlations) / np.sqrt(len(all_correlations)))

    df[f"EBS-ALS-{parameterization}-{c_estimation}"] = [f"{m:.2f} pm {s:.2f}" for (m,s) in zip(means, se)]
    
    params_combinations = corrections_dict["12-Tricosanone"].keys()
    correlations_dict = {}
    for params in params_combinations:
        all_correlations = []
        for compound in compounds:
            all_correlations.append(compute_cors(compound,params,ref_dict,corrections_dict,wn))
        all_correlations = np.concatenate(all_correlations)
        correlations_dict[params] = np.mean(all_correlations)
    
    selected_params = max(correlations_dict, key=correlations_dict.get)
    print(f"EBS-ALS-{parameterization}-* selected parameters:", selected_params)
    
    params = selected_params
    means = []
    se = []
    all_correlations = []

    for compound in compounds:
        correlations = compute_cors(compound,params,ref_dict,corrections_dict,wn)
        all_correlations.append(correlations)
        means.append(100 * np.mean(correlations))
        se.append(100 * np.std(correlations) / np.sqrt(len(correlations)))
    all_correlations = np.concatenate(all_correlations)
    means.append(100 * np.mean(all_correlations))
    se.append(100 * np.std(all_correlations) / np.sqrt(len(all_correlations)))

    df[f"EBS-ALS-{parameterization}-*"] = [f"{m:.2f} pm {s:.2f}" for (m,s) in zip(means, se)]
    df = pd.DataFrame(df).T
    df.columns = compounds + ["All"]
    
    return df


In [278]:
with open(CORRECTIONS_DIR / "ebs_als_dict_V.pkl", "rb") as f:
    corrections_dict = pickle.load(f)
    
ebs_metrics(corrections_dict, ref_dict, wn, chat=chat_ind, compounds = compounds, parameterization = "V", c_estimation="IND")

EBS-ALS-V-* selected parameters: ncomp,tau = 13,0.025


,12-Tricosanone,Ammonium sulfate,Malonic Acid,Suberic Acid,D-Glucose,fructose,levoglucosan,All
EBS-ALS-V-IND,64.67 pm 0.14,63.06 pm 0.40,62.20 pm 0.53,58.79 pm 0.29,38.65 pm 0.56,50.14 pm 0.23,48.57 pm 0.46,57.60 pm 0.57
EBS-ALS-V-*,86.39 pm 0.67,91.59 pm 0.20,70.57 pm 2.69,78.42 pm 1.51,75.57 pm 0.89,79.04 pm 1.05,80.71 pm 2.86,83.35 pm 0.81


In [279]:
with open(CORRECTIONS_DIR / "ebs_als_dict_W.pkl", "rb") as f:
    corrections_dict = pickle.load(f)
    
ebs_metrics(corrections_dict, ref_dict, wn, chat=chat_cv, compounds = compounds, parameterization = "W", c_estimation="CV")

EBS-ALS-W-* selected parameters: ncomp,tau = 13,0.025


,12-Tricosanone,Ammonium sulfate,Malonic Acid,Suberic Acid,D-Glucose,fructose,levoglucosan,All
EBS-ALS-W-CV,66.77 pm 0.10,63.49 pm 0.39,63.88 pm 0.40,61.11 pm 0.25,39.23 pm 0.64,53.53 pm 0.19,50.93 pm 0.59,59.38 pm 0.55
EBS-ALS-W-*,86.26 pm 0.78,91.63 pm 0.19,70.98 pm 2.66,78.12 pm 1.56,75.22 pm 0.81,79.07 pm 1.05,80.54 pm 2.82,83.27 pm 0.81
